In [ ]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader

In [ ]:
#device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
#print(f"Active device: {device}")

In [ ]:
df = pd.read_csv('fashion-mnist_train.csv')
X = df.iloc[:, 1:].values
y = df.iloc[:, 0].values

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size= 0.2)

In [ ]:
X_train = X_train / 255.0
X_test = X_test / 255.0

In [ ]:
class CustomDataset(Dataset):
  def __init__(self, features, labels):
    self.features = torch.tensor(features, dtype=torch.float32)
    self.labels = torch.tensor(labels, dtype=torch.long)

  def __len__(self):
    return len(self.features)

  def __getitem__(self, index):
    return self.features[index], self.labels[index]

In [ ]:
train_dataset = CustomDataset(X_train, y_train)
test_dataset = CustomDataset(X_test, y_test)

In [ ]:
class MyNN(nn.Module):
    def __init__(self, input_dim, output_dim, neurons_per_layer, num_hidden_layers, dropout_rate):
        super().__init__()
        layers = []
        for i in range(num_hidden_layers):
            layers.append(nn.Linear(input_dim, neurons_per_layer))
            layers.append(nn.BatchNorm1d(neurons_per_layer))
            layers.append(nn.ReLU())
            layers.append((nn.Dropout(dropout_rate)))
            input_dim = neurons_per_layer

        layers.append(nn.Linear(neurons_per_layer, output_dim))

        self.model = nn.Sequential(*layers)

    def forward(self, x):
        return self.model(x)

In [ ]:
def objective(trial):
    num_hidden_layers = trial.suggest_int('num_hidden_layers', 1, 5)
    neurons_per_layer = trial.suggest_int('neurons_per_layer', 8, 128, step=8)
    epochs = trial.suggest_int('epochs', 10, 50, step=10)
    learning_rate = trial.suggest_float('learning_rate', 1e-5, 1e-1, log=True)
    dropout_rate = trial.suggest_float('dropout_rate', 0.1, 0.5, step=0.1)
    batch_size = trial.suggest_categorical('batch_size', [16, 32, 64, 128])
    optimizer_name = trial.suggest_categorical('optimizer', ['SGD', 'Adam', 'RMSprop'])
    weight_decay = trial.suggest_float('weight_decay', 1e-5, 1e-1, log=True)

    train_loader = DataLoader(train_dataset, batch_size= batch_size, shuffle = True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle = False)



    input_dim = 784
    output_dim = 10

    model = MyNN(input_dim, output_dim, neurons_per_layer, num_hidden_layers, dropout_rate)
    criterion = nn.CrossEntropyLoss()
    if optimizer_name == 'Adam':
        optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay= weight_decay)
    elif optimizer_name == 'SGD':
        optimizer = optim.SGD(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    elif optimizer_name == 'RMSprop':
        optimizer = optim.RMSprop(model.parameters(), lr=learning_rate, weight_decay=weight_decay)


    for epoch in range(epochs):

      for batch_features, batch_labels in train_loader:

            #batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)
            outputs = model(batch_features)

            loss = criterion(outputs, batch_labels)

            optimizer.zero_grad()
            loss.backward()

            optimizer.step()

    model.eval()
    total = 0
    correct = 0

    with torch.no_grad():
      for batch_features, batch_labels in test_loader:
          #batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)

          output = model(batch_features)
          _, predicted = torch.max(output, 1)
          total += batch_labels.shape[0]

          correct += (predicted == batch_labels).sum().item()

    accuracy = correct/total

    return accuracy


In [ ]:
import optuna

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=10)

In [ ]:
study.best_params

In [ ]:
study.best_value